# Clustering audio à partir des fichiers chroma

On utilise ici les features de Chordino qui permettent d'étudier les accords.

In [77]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import seaborn as sns
import plotly.express as px
from plotly.offline import plot
import plotly.io as pio
from sklearn.manifold import MDS
import librosa
from scipy.cluster.hierarchy import linkage, dendrogram
import os
import importlib
import oeuvre
importlib.reload(oeuvre)
from oeuvre import Oeuvre

In [78]:
dico_res = {"maj":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0],
"min":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0],
"dim_dim7":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,1,0,0],
"dim_min7":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,1,0],
"maj_min7":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,1,0],
"maj_maj7":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,1],
"min_min7":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,1,0],
"dim":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0],
"aug":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0]}

In [79]:
dico_notes = {
    "A" : 0,
    "A#" : 1,
    "Bb" : 1,
    "B" : 2,
    "C" : 3,
    "C#" : 4,
    "Db" : 4,
    "D" : 5,
    "D#" : 6,
    "Eb" : 6,
    "E" : 7,
    "F" : 8,
    "F#" : 9,
    "Gb" : 9,
    "G" : 10,
    "G#" : 11,
    "Ab" : 11,
    "N" : -1
}

In [ ]:
li_filenames = []

for racine, _, fichiers in os.walk('cross-era_chords-chordino'):
    for fichier in fichiers:
        chemin_relatif = os.path.relpath(os.path.join(racine, fichier))
        li_filenames.append(chemin_relatif)

cross-era_chords-chordino\chords-chordino_orchestra_addon.csv
cross-era_chords-chordino\chords-chordino_orchestra_baroque.csv
cross-era_chords-chordino\chords-chordino_orchestra_classical.csv
cross-era_chords-chordino\chords-chordino_orchestra_modern.csv
cross-era_chords-chordino\chords-chordino_orchestra_romantic.csv
cross-era_chords-chordino\chords-chordino_piano_addon.csv
cross-era_chords-chordino\chords-chordino_piano_baroque.csv
cross-era_chords-chordino\chords-chordino_piano_classical.csv
cross-era_chords-chordino\chords-chordino_piano_modern.csv
cross-era_chords-chordino\chords-chordino_piano_romantic.csv


## Récupération d'un fichier et transformation en features

In [81]:
def get_matrix(chunk):
    li = []
    for accord in chunk:
        partition = accord.split("_")
        note = partition[0]
        if dico_notes[note] > 0:
            liste = [0]*12
            liste[dico_notes[note]]=1
            acc = "_".join(partition[1:])
            liste += dico_res[acc]
            li.append(liste)
    return np.array(li)

In [82]:
def get_chunks(filename):
    data = pd.read_csv(filename, sep=",", header=None)
    chunks = {}
    chunk = []
    title = None
    for i, row in data.iterrows():
        if pd.notna(row[0]):
            if title is not None:
                chunks[title] = chunk
            title = row[0]
            chunk = []
        else:
            chunk.append(row[2])
    
    if title is not None:
        chunks[title] = chunk
    return chunks

def dico_chunks(filename):
    chunks = get_chunks(filename)
    return {k: get_matrix(v) for k, v in chunks.items() if len(v) > 0}

## Création des objets

In [83]:
def construct_oeuvre(name,matrix = None):
    oeuvre = Oeuvre(name)
    oeuvre.filename = name
    name = name[:-3]
    part1 = name.split("/")
    cat = part1[0].split("_")
    oeuvre.category = cat[0]
    oeuvre.genre = cat[1]
    part2 = part1[1].split("_")[1:]
    oeuvre.compositeur = part2[0]
    oeuvre.titre = " ".join(part2[1:])
    oeuvre.matrix = matrix
    return oeuvre

In [84]:
print(construct_oeuvre("piano_classical/CrossEra-1001_Cimarosa_Piano_sonata_no._10_in_b-flat_major___allegro.mp3"))


        Titre : Piano sonata no. 10 in b-flat major   allegro.
        Compositeur : Cimarosa
        Genre : classical
        Catégorie : piano
        Fichier d'origine : piano_classical/CrossEra-1001_Cimarosa_Piano_sonata_no._10_in_b-flat_major___allegro.mp3
        


In [85]:
def li_oeuvres(filename):
    dico = dico_chunks(filename)
    return [construct_oeuvre(k,v) for k,v in dico.items()]

def encyclo_oeuvres(filenames):
    li = []
    for filename in filenames:
        li += li_oeuvres(filename)
        print(f"{filename} fait")
    return li

In [94]:
liste_oeuvres = encyclo_oeuvres(li_filenames)

cross-era_chords-chordino\chords-chordino_orchestra_addon.csv fait
cross-era_chords-chordino\chords-chordino_orchestra_baroque.csv fait
cross-era_chords-chordino\chords-chordino_orchestra_classical.csv fait
cross-era_chords-chordino\chords-chordino_orchestra_modern.csv fait
cross-era_chords-chordino\chords-chordino_orchestra_romantic.csv fait
cross-era_chords-chordino\chords-chordino_piano_addon.csv fait
cross-era_chords-chordino\chords-chordino_piano_baroque.csv fait
cross-era_chords-chordino\chords-chordino_piano_classical.csv fait
cross-era_chords-chordino\chords-chordino_piano_modern.csv fait
cross-era_chords-chordino\chords-chordino_piano_romantic.csv fait


## Clustering